In [ ]:
import numpy as np
import pandas as pd
from numpy.random import seed
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
import scipy.stats as stats
from collections import Counter

mut = pd.read_csv('MUTATIONS.csv')
mut = mut[mut.IS_SYNONYMOUS == False ]

drug_names = ['AMIKACIN','ETHAMBUTOL', 'ETHIONAMIDE', 'ISONIAZID',\
              'KANAMYCIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN', 'RIFAMPICIN', 'RIFABUTIN']
n_drugs = len(drug_names) #number of drugs 

accs = {}

for h in [0.25, 0.5, 0.75, 0.99]: #test size
    for P in [0.05, 0.2, 0.4, 0.6, 0.8, 0.99]: #drop rate
        for v in range(5): #seeds
            seed(v)

            x = pd.read_csv('GAM_train.csv', index_col = ['ID']).to_numpy()
            y = pd.read_csv('Y9_train.csv').to_numpy()

            x1 = pd.read_csv('val_X.csv', index_col = ['ID']).to_numpy()
            y1 = pd.read_csv('valYs.csv').to_numpy()

            x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = h)
            
            x_train = np.concatenate((x_train,x1))
            y_train = np.concatenate((y_train,y1))

            y_train = pd.DataFrame(y_train).set_index(0)
            y_train = y_train.to_numpy().astype('int8')

            clf = GradientBoostingClassifier()
            mlf = MultiOutputClassifier(clf)
            mlf.fit(x_train,y_train)

            yt = pd.DataFrame(y_test).set_index(0)
            y_test = yt.to_numpy().astype('int8')
            yt = yt.index
            acc = mlf.score(x_test,y_test)
            accs[str(h*100)+'_'+str(P*100)+'_'+str(v)] = acc
            
            matrix = np.random.choice([0, 1], size=(len(y_test), 9), p=[P, 1-P])
            y_test = pd.DataFrame(y_test).replace([0,1],[1,2])
            y_test = y_test*matrix
            y_test = y_test.replace([1,2,0],[0,1,'a']).to_numpy()

            yhat = mlf.predict(x_test)
            yhat = yhat.round()

            for i in range(len(y_test)):
                for j in range(len(y_test[i])):
                    if y_test[i][j] == "a":
                        y_test[i][j] = yhat[i][j]

            drugs = pd.DataFrame(y_test, index=yt , columns=drug_names).replace([0,1],['S','R'])

            y = []
            for i in range(len(drugs)):
                x = drugs.iloc[i,:] #taking resitence profile for each strain
                y.append(str(list(x))) #profile added to list

            counts = Counter(y) #count the number of time each unique profile occurs
            r = pd.Series(counts)

            r = r.sort_values(ascending = False) #sort largest to smallest
            indexs = r.index #list of unique resistence profile 

            pval = {}
            for i in range(len(indexs)): #filters for each drug and places them into approprate group 
                S = indexs[i]
                fix = drugs[drugs.AMIKACIN == S[2] ]
                fix = fix[fix.ETHAMBUTOL == S[7] ]
                fix = fix[fix.ETHIONAMIDE == S[12] ]
                fix = fix[fix.ISONIAZID == S[17] ]
                fix = fix[fix.KANAMYCIN == S[22] ]
                fix = fix[fix.LEVOFLOXACIN == S[27] ]
                fix = fix[fix.MOXIFLOXACIN == S[32] ]
                fix = fix[fix.RIFAMPICIN == S[37] ]
                fix = fix[fix.RIFABUTIN == S[42] ]

                if len(fix) > 1:
                    listFix = list(fix.index)

                    if S == "["+"'S', "*(n_drugs-1)+"'S']": #control
                        lenS = len(fix)
                        mutS = mut[mut.UNIQUEID.isin(listFix)]  

                        ENA_snps = {}
                        for k in range(lenS):
                            folderK = listFix[k]
                            snps = mutS[mutS.UNIQUEID.isin([folderK])]
                            ENA_snps[folderK] = snps.groupby(['MUTATION','GENE'])['MUTATION'].count()

                        ENA_snps = pd.DataFrame(ENA_snps)
                        SMs = ENA_snps.sum(axis=1) # numebr of time mutation occurs in control 

                    else: #test groups
                        lenX = len(fix)
                        mutX = mut[mut.UNIQUEID.isin(listFix)]

                        ENA_snps = {}
                        for k in range(lenX):
                            folderK = listFix[k]
                            snps = mutX[mutX.UNIQUEID.isin([folderK])]
                            ENA_snps[folderK] = snps.groupby(['MUTATION','GENE'])['MUTATION'].count()

                        ENA_snps = pd.DataFrame(ENA_snps)
                        RMs = ENA_snps.sum(axis=1) # numebr of time mutation occurs in test group


                        df = pd.concat([RMs,SMs],axis=1).replace(np.nan, 0)
                        df = df[df[0] > 0]
                        for j in range(0,len(df)):
                            data = [[ df.iloc[j,0], df.iloc[j,1]] , [ lenX-df.iloc[j,0], lenS-df.iloc[j,1] ]]
                            odd_ratio, p_value = stats.fisher_exact(data)
                            df.iloc[j,0] = odd_ratio
                            df.iloc[j,1] = p_value

                        if len(df) > 0:
                            bon = 0.05/len(df)
                            df = df[df.iloc[:,1] <= bon]  #filter out all non-significant mutations
                            df = df[df.iloc[:,0] > 1] #filter out mutations that are correlated negativly

                            pval[S] = df.iloc[:,1]

            if len(pval) != 0:
                pval = pd.DataFrame(pval).replace(np.nan, 1)
                MD_pval = pval.iloc[:,0:n_drugs] #to be writen over
                
                for i in range(len(pval)):
                    A = [0] * n_drugs
                    B = [0] * n_drugs
                    C = [0] * n_drugs
                    D = [0] * n_drugs
                    mut_pval = pval.iloc[i] 

                    for j in range(len(mut_pval)):
                        drug_set = mut_pval.index[j]

                        if mut_pval[j] < 0.05: 
                            for k in range(2, len(drug_set), 5):
                                if drug_set[k] == 'S': 
                                    C[int((k-2)/5)] += 1 
                                elif drug_set[k] == 'R': 
                                    A[int((k-2)/5)] += 1 

                        else:
                            for k in range(2, len(drug_set), 5):
                                if drug_set[k] == 'S': 
                                    D[int((k-2)/5)] += 1 
                                elif drug_set[k] == 'R': 
                                    B[int((k-2)/5)] += 1 

                    for q in range(n_drugs):
                        data = [ [A[q],B[q]], [C[q],D[q]] ]
                        odd_ratio, p_value = stats.fisher_exact(data)

                        if odd_ratio > 1:
                            MD_pval.iloc[i,q] = p_value
                        else:
                            MD_pval.iloc[i,q] = 1
                
                MDbon = 0.05/len(MD_pval)
                MD_pval = MD_pval[MD_pval <= MDbon]
                MD_pval.dropna(axis=0, how ='all', inplace=True)
                MD_pval = MD_pval.set_axis(drug_names,axis=1)
                print(MD_pval)
   
            else:
                print(str(h*100)+'_'+str(P*100)+'_'+str(v)+'fail :(')
            
accs = pd.Series(accs)